# 第 20 节：PPO 从零完整实现

## 📍 位置
PPO 理论 (19) → **PPO 从零实现 (20)** → 调试 (21) → ...

这是整个课程中**最重要的实践 Notebook**。

## 🎯 学习目标
1. 从零实现完整的 PPO 算法
2. 理解每个组件的用途和相互作用
3. 掌握 GAE、clipping、多轮更新
4. 学会监控训练指标
5. 在 CartPole 上获得稳定训练结果

## 1. PPO 完整架构回顾

### 核心公式
$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t)\right]$$

其中 $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$

### 完整流程
1. 收集 $N$ 步 rollout 数据
2. 用 GAE 计算 advantage
3. 多轮 minibatch SGD 更新
4. 重复

In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np; import torch; import torch.nn as nn; import torch.optim as optim
import gymnasium as gym; import matplotlib.pyplot as plt
import os; from collections import deque; from typing import List, Tuple
FIG_DIR = 'outputs/figures'; CKPT_DIR = 'outputs/checkpoints'
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


## 2. 网络定义（复用 rl_course）

In [ ]:
from rl_course.networks.mlp import ActorCriticNetwork
from rl_course.buffers.rollout_buffer import RolloutBuffer
print('✅ 网络和缓冲区已导入')

## 3. PPO Agent 从零手写

虽然我们有 `rl_course.agents.ppo.PPOAgent`，但为了深入理解每个细节，
我们在 Notebook 中**手动实现训练循环**，直接调用网络和 buffer。

In [ ]:
# === 超参数（快速模式）===
class PPOConfig:
    state_dim = 4; n_actions = 2
    hidden_dims = [64, 64]
    gamma = 0.99; gae_lambda = 0.95; clip_epsilon = 0.2
    c1 = 0.5; c2 = 0.01  # value loss, entropy coefficients
    lr = 3e-4; n_steps = 512; batch_size = 64
    n_epochs = 10; max_grad_norm = 0.5; target_kl = 0.015
    total_timesteps = 50_000  # 快速模式

# === 初始化 ===
network = ActorCriticNetwork(PPOConfig.state_dim, PPOConfig.n_actions, PPOConfig.hidden_dims).to(DEVICE)
optimizer = optim.Adam(network.parameters(), lr=PPOConfig.lr)
buffer = RolloutBuffer(PPOConfig.n_steps, PPOConfig.state_dim, PPOConfig.gamma, PPOConfig.gae_lambda)

# === 训练指标跟踪 ===
episode_returns, policy_losses, value_losses, entropies, clip_fractions, approx_kls = [], [], [], [], [], []

print(f"PPO config: {PPOConfig.n_steps} steps/rollout, {PPOConfig.n_epochs} epochs, clip={PPOConfig.clip_epsilon}")

## 4. 数据收集函数

In [ ]:
def collect_rollout(env, network, buffer, n_steps, state, episode_return=0):
    """Collect n_steps of experience.

    Args:
        state: Initial state (inherited from previous rollout).

    Returns:
        episode_returns: List of completed episode returns in this rollout.
        state: Final state (for GAE bootstrap and next rollout).
    """
    episode_returns = []

    for step in range(n_steps):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        action, log_prob, value = network.get_action(state_t)

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        # Distinguish terminated from truncated for GAE bootstrap mask
        # Compute next_value BEFORE env.reset()
        with torch.no_grad():
            nxt_st = torch.FloatTensor(next_state).unsqueeze(0).to(DEVICE)
            next_value = network.get_value(nxt_st).item()

        buffer.add(state, action.item(), reward, done, log_prob.item(), value.item(), next_value, terminated=terminated)
        episode_return += reward
        state = next_state

        if done:
            episode_returns.append(episode_return)
            episode_return = 0
            state, _ = env.reset()

    return episode_returns, state, episode_return


## 5. PPO 更新函数（核心）

In [ ]:
def ppo_update(network, optimizer, buffer):
    """PPO update: GAE computation + multi-epoch minibatch SGD.

    KL early stopping: after each epoch, compute full-batch KL.
    If KL exceeds target_kl, stop remaining epochs.

    Returns:
        dict with averaged metrics.
    """
    # 1. Compute GAE using stored next_values
    buffer.compute_gae()

    # 2. Normalize FULL batch BEFORE creating minibatches
    n = buffer.size
    adv_t = torch.FloatTensor(buffer.advantages[:n]).to(DEVICE)
    adv_t = (adv_t - adv_t.mean()) / (adv_t.std(unbiased=False) + 1e-8)
    buffer.advantages[:n] = adv_t.cpu().numpy()

    total_policy_loss, total_value_loss, total_entropy = 0.0, 0.0, 0.0
    total_clip_frac, total_approx_kl = 0.0, 0.0
    n_updates = 0
    early_stopped = False
    epochs_completed = 0
    final_full_batch_kl = None

    for epoch in range(PPOConfig.n_epochs):
        # Reshuffle minibatches each epoch for diversity
        batches = buffer.get_minibatches(PPOConfig.batch_size, shuffle=True)
        for states, actions, returns, advantages, old_log_probs, old_values in batches:
            # Advantages already normalized at full-batch level

            # Forward pass
            logits, values = network(states)
            values = values.squeeze(-1)  # (batch,)

            # === Policy Loss (PPO-Clip) ===
            new_log_probs = torch.log_softmax(logits, dim=-1)  # (batch, n_actions)
            action_log_probs = new_log_probs.gather(1, actions.unsqueeze(-1)).squeeze(-1)  # (batch,)

            ratio = torch.exp(action_log_probs - old_log_probs)  # (batch,) importance ratio

            # Clipped surrogate objective
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - PPOConfig.clip_epsilon, 1 + PPOConfig.clip_epsilon) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            # === Clipped Value Loss ===
            v_clipped = old_values + torch.clamp(values - old_values, -PPOConfig.clip_epsilon, PPOConfig.clip_epsilon)
            value_loss1 = (values - returns) ** 2
            value_loss2 = (v_clipped - returns) ** 2
            value_loss = 0.5 * torch.max(value_loss1, value_loss2).mean()

            # === Entropy Bonus ===
            probs = torch.softmax(logits, dim=-1)
            log_probs_all = torch.log_softmax(logits, dim=-1)
            entropy = -(probs * log_probs_all).sum(dim=-1).mean()  # scalar

            # === Total Loss ===
            total_loss = policy_loss + PPOConfig.c1 * value_loss - PPOConfig.c2 * entropy

            # === Metrics (pre-step snapshot) ===
            with torch.no_grad():
                clip_frac = ((ratio < 1 - PPOConfig.clip_epsilon) | (ratio > 1 + PPOConfig.clip_epsilon)).float().mean().item()
                approx_kl = (old_log_probs - action_log_probs).mean().item()

            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            total_entropy += entropy.item()
            total_clip_frac += clip_frac
            total_approx_kl += approx_kl
            n_updates += 1

            # === Backward ===
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), PPOConfig.max_grad_norm)
            optimizer.step()

        epochs_completed += 1

        # Post-epoch KL check: use UPDATED network to compute full-batch KL
        if PPOConfig.target_kl is not None:
            with torch.no_grad():
                all_states_t = torch.FloatTensor(buffer.states[:n]).to(DEVICE)
                all_actions_t = torch.LongTensor(buffer.actions[:n]).to(DEVICE)
                all_old_log_probs_t = torch.FloatTensor(buffer.log_probs[:n]).to(DEVICE)

                all_logits, _ = network(all_states_t)
                all_new_log_probs_all = torch.log_softmax(all_logits, dim=-1)
                all_action_log_probs = all_new_log_probs_all.gather(1, all_actions_t.unsqueeze(-1)).squeeze(-1)

                # Unbiased KL estimator
                log_ratio_full = all_action_log_probs - all_old_log_probs_t
                ratio_full = torch.exp(log_ratio_full)
                final_full_batch_kl = ((ratio_full - 1.0) - log_ratio_full).mean().item()

            if final_full_batch_kl > PPOConfig.target_kl:
                early_stopped = True
                break  # Stop remaining epochs

    n_updates = max(n_updates, 1)  # guard against division by zero
    return {
        'policy_loss': total_policy_loss / n_updates,
        'value_loss': total_value_loss / n_updates,
        'entropy': total_entropy / n_updates,
        'clip_frac': total_clip_frac / n_updates,
        'approx_kl': total_approx_kl / n_updates,
        'epochs_completed': epochs_completed,
        'early_stopped': early_stopped,
        'final_full_batch_kl': final_full_batch_kl,
    }


## 6. 训练循环

In [ ]:
env = gym.make("CartPole-v1")
state, _ = env.reset()
episode_return = 0

for update_idx in range(PPOConfig.total_timesteps // PPOConfig.n_steps):
    # Phase 1: data collection
    buffer.reset()
    ep_returns, state, episode_return = collect_rollout(env, network, buffer, PPOConfig.n_steps, state, episode_return)

    # Phase 2: PPO update (pass last_state for GAE bootstrap)
    metrics = ppo_update(network, optimizer, buffer)

    # Record metrics
    episode_returns.extend(ep_returns)
    policy_losses.append(metrics['policy_loss'])
    value_losses.append(metrics['value_loss'])
    entropies.append(metrics['entropy'])
    clip_fractions.append(metrics['clip_frac'])
    approx_kls.append(metrics['approx_kl'])

    if (update_idx + 1) % 10 == 0:
        print(f"Update {update_idx+1:3d} | "
              f"KL={metrics['approx_kl']:.4f} | "
              f"Clip%={metrics['clip_frac']:.2%} | "
              f"Ent={metrics['entropy']:.3f}")

print()
print(f"Training complete! Total timesteps: {PPOConfig.total_timesteps:,}")


## 7. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Episode Return
axes[0,0].plot(episode_returns, linewidth=0.5, alpha=0.7, color='steelblue')
axes[0,0].set_title('Episode Return'); axes[0,0].set_xlabel('Update'); axes[0,0].grid(True, alpha=0.3)

# Policy Loss
axes[0,1].plot(policy_losses, linewidth=0.5, alpha=0.7, color='coral')
axes[0,1].set_title('Policy Loss'); axes[0,1].set_xlabel('Update'); axes[0,1].grid(True, alpha=0.3)

# Value Loss
axes[0,2].plot(value_losses, linewidth=0.5, alpha=0.7, color='purple')
axes[0,2].set_title('Value Loss'); axes[0,2].set_xlabel('Update'); axes[0,2].grid(True, alpha=0.3)

# Entropy
axes[1,0].plot(entropies, linewidth=0.5, alpha=0.7, color='green')
axes[1,0].set_title('Entropy'); axes[1,0].set_xlabel('Update'); axes[1,0].grid(True, alpha=0.3)

# Clip Fraction
axes[1,1].plot(clip_fractions, linewidth=0.5, alpha=0.7, color='orange')
axes[1,1].axhline(y=0.1, color='r', linestyle='--', alpha=0.5)
axes[1,1].set_title('Clip Fraction'); axes[1,1].set_xlabel('Update'); axes[1,1].grid(True, alpha=0.3)

# Approx KL
axes[1,2].plot(approx_kls, linewidth=0.5, alpha=0.7, color='brown')
axes[1,2].axhline(y=PPOConfig.target_kl, color='r', linestyle='--', alpha=0.5, label='target_kl')
axes[1,2].set_title('Approx KL'); axes[1,2].set_xlabel('Update'); axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/20_ppo_training.png', dpi=100); plt.close()
print("✅ PPO 训练曲线已保存")

## 8. 保存和加载模型

In [ ]:
torch.save({'network': network.state_dict(), 'optimizer': optimizer.state_dict(), 'config': PPOConfig},
           f'{CKPT_DIR}/ppo_cartpole.pt')
print(f"✅ 模型已保存到 {CKPT_DIR}/ppo_cartpole.pt")

## 9. 评估和录制

In [ ]:
def evaluate(network, env, n_episodes=10):
    returns = []
    for _ in range(n_episodes):
        state, _ = env.reset(); done = False; ep_ret = 0
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            action, _, _ = network.get_action(state_t, deterministic=True)
            state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated; ep_ret += reward
        returns.append(ep_ret)
    return np.mean(returns), np.std(returns)

mean_r, std_r = evaluate(network, env, n_episodes=10)
print(f"Evaluation (10 episodes): {mean_r:.1f} ± {std_r:.1f}")

# 录制视频
env_render = gym.make("CartPole-v1", render_mode="rgb_array")
from rl_course.visualization.video import record_episode

def policy_fn(state):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
    action, _, _ = network.get_action(state_t, deterministic=True)
    return action.item()

record_episode(env_render, policy_fn, filepath='outputs/videos/20_ppo_cartpole.gif', fps=20, max_steps=500)
env_render.close()
print("✅ 演示视频已保存")

## 10. 使用 rl_course 封装版 PPO（对比）

In [ ]:
from rl_course.agents.ppo import PPOAgent
from rl_course.utils.seeding import set_seed; set_seed(42)

agent = PPOAgent(state_dim=4, n_actions=2, hidden_dims=[64,64], n_steps=512, batch_size=64, n_epochs=10)
env2 = gym.make("CartPole-v1")

# 使用封装好的 PPO 训练（少量步数做对比）
n_total = 10000
state, _ = env2.reset(); ep_ret = 0
returns2 = []

for step in range(n_total):
    action = agent.act(state, train=True)
    next_state, reward, terminated, truncated, _ = env2.step(action)
    done = terminated or truncated

    # Compute next_value BEFORE env.reset()
    nxt_st = torch.FloatTensor(next_state).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        next_value = agent.network.get_value(nxt_st).item()

    agent.store(reward=reward, done=terminated or truncated, terminated=terminated, next_value=next_value)
    ep_ret += reward
    state = next_state

    if done:
        state, _ = env2.reset()
        returns2.append(ep_ret)
        ep_ret = 0

    if agent.buffer.size >= agent.buffer.buffer_size:
        agent.update()

    if (step + 1) % 5000 == 0:
        recent = np.mean(returns2[-10:]) if returns2 else 0
        print(f"Step {step+1:6d} | Avg Return: {recent:.1f}")

print(f"\\nrl_course PPO 最终表现: {np.mean(returns2[-10:]):.1f}" if returns2 else "\\n未完成任何 episode")
env2.close()


## 11. 关键指标解读

### Clip Fraction
- **含义**：被裁剪到 [1-ε, 1+ε] 范围外的比率占比。低 clip fraction 意味着大部分更新在裁剪范围内（策略变化较小）。
- **经验参考（当前实验）**：0.01-0.1
- **过低（接近 0）**：策略几乎没有变化，常见原因包括学习率偏小、更新 epoch 太少、梯度较小或 ε 过宽。建议检查学习率是否足够，尝试增加 epoch 数或适当减小 ε。
- **过高（> 0.2-0.3）**：太多更新被裁剪，ε 相对于当前策略变化幅度可能偏小。建议适当增大 ε 或减小学习率。
- **注意**：clip fraction 的"理想范围"高度依赖于具体任务和超参数配置，以上数值仅作为当前实验的经验参考，不能作为通用正确性标准。

### Approx KL
- **理想趋势**：逐渐下降或稳定
- **持续上升**：学习率太大或 ε 太大
- **突然跳跃**：策略可能崩溃
- **注意**：KL early stopping 是可选的安全措施，不是 PPO 正确性的必要组件

### Entropy
- **理想趋势**：缓慢下降
- **太快降到接近 0**：策略过早收敛，可能卡在局部最优
- **一直很高**：策略没学到任何东西

### Explained Variance
$$\text{EV} = 1 - \frac{\text{Var}(G - V)}{\text{Var}(G)}$$
- 衡量 Critic 预测的准确度
- **接近 1**：Critic 能很好地解释回报变化
- **接近 0（或负）**：Critic 毫无用处

## 12. 总结

PPO 完整实现包含：
1. Actor-Critic 网络
2. Rollout 数据收集
3. GAE 优势估计
4. PPO-Clip 替代目标
5. Clipped Value Loss
6. Entropy Bonus
7. 多轮 Minibatch 更新
8. KL 早停
9. 梯度裁剪
10. Advantage 标准化

## 13. 练习
1. 修改 `clip_epsilon` (0.1, 0.2, 0.3)，观察 clip fraction 变化
2. 去掉 clipped value loss，只用 MSE，对比训练稳定性
3. 将 `n_epochs` 从 10 改为 1，观察性能变化
4. 在 Acrobot-v1 上测试 PPO
5. 实现 learning rate annealing

---
*下一节：[21_rl_debugging.ipynb](21_rl_debugging.ipynb) — RL 调试方法论*